# 03 — From Tiling to Crease Pattern (Shrink–Rotate)


The algorithm can be described in three steps:

1. Compute a **reciprocal figure**: For each face of the input graph, find a point so that the segment connecting two such points across an edge is perpendicular to that edge ([Maxwell–Cremona-style stress diagram](https://en.wikipedia.org/wiki/Cremona_diagram)). The position is stored as `face['reciprocal_pos']`.
2. **Shrink-rotate** every face: Shrink each face by a given factor, and rotate around its reciprocal point. The default scale factor is 0.5 and the rotation is $\\pi/5$, but each is a free parameter.
3. **Connect** corners of these faces which were touching in the original tiling (before being shrunk) by creases. 

However, the implementation actually performs step (3) before step (2), via the ``shrink_rotate`` conway operator.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    shrink_rotate,
    rendering,
)
from eucare.rendering import multi_show


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.shrink_rotate import assign_this_way_by_face_z_order, shrink_rotate_pattern


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = shrink_rotate_pattern(G)
    SRG.recompute_lengths_and_angles()
    return SRG


## Build a square tiling and its SRG

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)
multi_show([G, SRG],
           titles=['input tiling', 'shrink-rotate CP'],
           face_inset=0.04, render_vertices=False)


## Inspecting the reciprocal figure

After step 1, every face of `G` carries an extra `face['reciprocal_pos']`. Below we mark these points (red) on top of the input tiling — these are the centres around which step 2 rotates each face.

In [ ]:
from eucare.shrink_rotate import reciprocal_figure
G2 = G.copy()
G2.recompute_lengths_and_angles()
# (re-run reciprocal_figure to ensure attribute is fresh)
reciprocal_figure(G2)

fig, ax = plt.subplots(figsize=(5, 5))
for h in G2.halfedges_representing_edges():
    a = G2.geometry.to_euclidean(h.orig['pos'])
    b = G2.geometry.to_euclidean(h.dest['pos'])
    ax.plot([a[0], b[0]], [a[1], b[1]], color='lightgray', linewidth=1)
for f in G2.faces:
    if 'reciprocal_pos' in f.attributes:
        p = G2.geometry.to_euclidean(f['reciprocal_pos'])
        ax.plot(p[0], p[1], 'o', color='crimson', markersize=4)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('reciprocal points (red) on input tiling')
plt.show()


## What does `z_order` actually decide?

The SRG construction itself does not need a face order. What the z-order chooses is the *orientation of every interior edge*: `assign_this_way_by_face_z_order` walks the halfedges and sets a boolean attribute `this_way=True` exactly on those that point from a higher-z face to a lower-z face.

This orientation drives the **mountain/valley** assignment baked in by `assign_shrink_rotate_creases` (called inside `shrink_rotate_pattern`). Different consistent face orderings therefore produce the same *geometry* but different *fold-up directions* — and so different valid crease patterns.

BFS from a central face (what `srg_pipeline` does) is the simplest choice. We can verify it leaves the cease assignments consistent:

In [ ]:
from eucare.overlap import CREASE_ASSIGNMENT, MOUNTAIN, VALLEY
n_m = sum(1 for h in SRG.halfedges_representing_edges()
          if h.attributes.get(CREASE_ASSIGNMENT) == MOUNTAIN)
n_v = sum(1 for h in SRG.halfedges_representing_edges()
          if h.attributes.get(CREASE_ASSIGNMENT) == VALLEY)
n_other = sum(1 for h in SRG.halfedges_representing_edges()
              if h.attributes.get(CREASE_ASSIGNMENT) not in (MOUNTAIN, VALLEY))
print(f'mountain={n_m}, valley={n_v}, unassigned/border={n_other}')


## Standard render preset in action

`shrink_rotate_pattern` already paints each crease with the right colour key (red for mountain, blue for valley). Combining that with `rendering.CREASE_PATTERN_PRESET` gives a clean default look:

In [ ]:
SRG.show(**rendering.CREASE_PATTERN_PRESET)


## What's next

- [`04_Folding_and_Overlap`](04_Folding_and_Overlap.ipynb) — solve for the folded face stacking order.
- [`Better reciprocal Figures`](Better%20reciprocal%20Figures.ipynb) (legacy) — more advanced examples with explicit reciprocal-figure construction.